# SAM 2.1 Hiera-Small — DIMER end-to-end promptable segmentation fine-tuning

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/sam2-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/sam2-segmentation-pipeline/blob/main/tutorials/sam2_segmentation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fsam2.1--hiera--small-ffcc4d?style=flat)](https://huggingface.co/facebook/sam2.1-hiera-small)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** pinned SAM 2.1 inference plus bounded mask-hypernetwork gradient adaptation, held-out mask-IoU evaluation, safe adapter export, and fresh reload

**This notebook is standalone.** It carries the repository's pipeline module (`src/sam2_segmentation_pipeline/pipeline.py` at revision `bf2ee2d972eb`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `ee5bba1d82bb8749febdf90f45e84b687142ba03` (~184 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** on a fresh CUDA runtime installs the pinned dependencies, verifies the exact base snapshot, generates and validates 24 records, freezes the base, performs the bounded gradient update, scores the held-out split, predicts an unseen scene, exports the adapter, reloads it over a fresh base, verifies numeric equivalence, and writes provenance without a clone, credential, upload, or configuration edit.

**Bring Your Own Data:** Set `USE_BYOD = True` to upload a bounded paired image/mask ZIP. The uploaded records pass through the same validation, split, adaptation, evaluation, export, and reload path as the generated dataset.

This notebook performs a real local gradient update. It freezes the Hiera encoder, prompt encoder, and the rest of the mask decoder, then trains only the first mask-token output hypernetwork (139,808 parameters) with BCE plus soft-Dice loss. The default dataset is 24 deterministic generated shape scenes split 18/6 before model execution. A SafeTensors adapter and closed SHA-256 manifest are exported and attached to a fresh pinned base model. These are tutorial sample-sanity results, not a segmentation benchmark.

**Learning objectives:** validate paired image/mask/prompt data, preserve a held-out split, measure the pretrained baseline, run bounded AdamW adaptation, evaluate mask IoU against a prompt-box baseline, infer on unseen data, export a base-bound SafeTensors adapter, and verify it after a fresh-model reload.

**This notebook does not demonstrate:** full-model training, video tracking, automatic segment-everything, semantic class learning, production-scale training, or benchmark claims. The adapter changes one image-mode mask hypernetwork only.

## Prerequisites

- **Runtime:** fresh Python 3.12 with an NVIDIA T4-class GPU or better. CUDA is required for the default fine-tuning path. The pinned 184 MB checkpoint is acquired automatically.
- **Knowledge:** Python, binary masks, IoU, train/validation separation, and adapter-versus-base semantics.
- **Data:** the default path generates 24 shape scenes. Optional BYOD accepts a ZIP with `images/` and `masks/` files paired by stem; it is off by default and follows the same E2E path. Do not upload confidential or restricted data unless you are authorized to use it in the hosted runtime.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/sam2.1-hiera-small` snapshot (~184 MB in total) at revision `ee5bba1d82bb…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `safetensors` versions, and whether CUDA is available.

In [1]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'sam2-segmentation-pipeline',
    'repository_revision': 'bf2ee2d972eb76fea165781ac1206bc65483b987',
    'embedded_module': 'src/sam2_segmentation_pipeline/pipeline.py',
    'embedded_modules': ['src/sam2_segmentation_pipeline/pipeline.py'],
    'module_sha256': '2b37878d8b05582ded0a0d5e48e867b73d7de933e5581b20c14bf8431f7ea24f',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, safetensors
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'safetensors': safetensors.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'sam2-segmentation-pipeline', 'repository_revision': 'bf2ee2d972eb76fea165781ac1206bc65483b987', 'embedded_module': 'src/sam2_segmentation_pipeline/pipeline.py', 'embedded_modules': ['src/sam2_segmentation_pipeline/pipeline.py'], 'module_sha256': '2b37878d8b05582ded0a0d5e48e867b73d7de933e5581b20c14bf8431f7ea24f', 'generator': 'build_notebook.py/2', 'notebook_spec': '2.0'}, 'python': '3.12.13', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'safetensors': '0.8.0', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/sam2_segmentation_pipeline/` @ `bf2ee2d972eb`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/sam2_segmentation_pipeline/pipeline.py`

In [2]:
from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "facebook/sam2.1-hiera-small"
MODEL_REVISION = "ee5bba1d82bb8749febdf90f45e84b687142ba03"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "sam2.1-hiera-small"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
ARTIFACT_FORMAT = "org.valcorza.sam2.mask-decoder-adapter"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_MANIFEST_NAME = "manifest.json"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
TRAINABLE_PREFIXES = ("mask_decoder.output_hypernetworks_mlps.0.",)
MAX_ADAPTATION_RECORDS = 128

# Input ceilings. The processor resizes every image to 1024x1024 (preprocessor_config.json), so model cost is
# fixed; the caller's resolution only sets the size of the up-sampled output masks. Prompts are one object per
# call: up to MAX_PROMPTS point clicks (label 1 = foreground, 0 = background) and/or one xyxy box.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
MAX_PROMPTS = 16
NUM_MULTIMASK_OUTPUTS = 3  # config.json mask_decoder_config.num_multimask_outputs
MASK_THRESHOLD = 0.0  # logits above this become True in the binarised masks (processor default)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def mask_iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection-over-union of two boolean masks of identical shape; the primitive behind any mIoU."""
    a = np.asarray(a)
    b = np.asarray(b)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")
    if a.dtype != np.bool_ or b.dtype != np.bool_:
        raise TypeError("mask_iou expects boolean arrays")
    union = np.logical_or(a, b).sum()
    return float(np.logical_and(a, b).sum() / union) if union else 0.0


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def validate_prompts(
    width: int,
    height: int,
    points: Sequence[Sequence[float]] | None,
    point_labels: Sequence[int] | None,
    box: Sequence[float] | None,
) -> tuple[list[list[float]] | None, list[int] | None, list[float] | None]:
    """Check one object's prompts: points inside the image with 0/1 labels, and/or one xyxy box inside it."""
    if points is None and box is None:
        raise ValueError("provide at least one of points or box")
    clean_points = clean_labels = None
    if points is not None:
        if isinstance(points, str) or not isinstance(points, Sequence):
            raise TypeError("points must be a sequence of [x, y] pairs")
        if not 1 <= len(points) <= MAX_PROMPTS:
            raise ValueError(f"point count {len(points)} outside 1..MAX_PROMPTS {MAX_PROMPTS}")
        if point_labels is None or len(point_labels) != len(points):
            raise ValueError("point_labels must be given with one 0/1 entry per point")
        clean_points, clean_labels = [], []
        for (x, y), label in zip(points, point_labels, strict=True):
            if not (0 <= x < width and 0 <= y < height):
                raise ValueError(f"point ({x}, {y}) outside image {width}x{height}")
            if label not in (0, 1) or isinstance(label, bool):
                raise ValueError(f"point label must be 0 or 1, got {label!r}")
            clean_points.append([float(x), float(y)])
            clean_labels.append(int(label))
    elif point_labels is not None:
        raise ValueError("point_labels given without points")
    clean_box = None
    if box is not None:
        if len(box) != 4:
            raise ValueError("box must be [x0, y0, x1, y1]")
        x0, y0, x1, y1 = (float(v) for v in box)
        if not (0 <= x0 < x1 <= width and 0 <= y0 < y1 <= height):
            raise ValueError(f"box {box!r} is not a non-empty xyxy box inside image {width}x{height}")
        clean_box = [x0, y0, x1, y1]
    return clean_points, clean_labels, clean_box


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one object's prompts: point clicks "
        "and/or one xyxy box"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "points": [1, MAX_PROMPTS],
    "point_labels": "one per point, 1 = foreground and 0 = background",
    "box": "at most one [x0, y0, x1, y1] inside the image with x0 < x1 and y0 < y1",
    "objects_per_call": 1,
    "multimask_outputs": NUM_MULTIMASK_OUTPUTS,
    "preprocessing": (
        "image converted to RGB; the processor resizes it to 1024x1024; returned masks are up-sampled "
        f"to the input resolution and binarised at logit MASK_THRESHOLD={MASK_THRESHOLD}"
    ),
}


def _check_inputs(
    image: Any,
    points: Any,
    point_labels: Any,
    box: Any,
    multimask: Any,
) -> tuple[Image.Image, list[list[float]] | None, list[int] | None, list[float] | None]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the cleaned request.

    ``segment`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    clean_points, clean_labels, clean_box = validate_prompts(
        rgb.width, rgb.height, points, point_labels, box
    )
    if not isinstance(multimask, bool):
        raise TypeError("multimask must be a bool")
    return rgb, clean_points, clean_labels, clean_box


def validate_inputs(
    image: Image.Image,
    *,
    points: Sequence[Sequence[float]] | None = None,
    point_labels: Sequence[int] | None = None,
    box: Sequence[float] | None = None,
    multimask: bool = True,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``segment`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, clean_points, clean_labels, clean_box = _check_inputs(
        image, points, point_labels, box, multimask
    )
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (segment takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
                "n_points": 0 if clean_points is None else len(clean_points),
                "has_box": clean_box is not None,
            }
        ],
        "points": clean_points,
        "point_labels": clean_labels,
        "box": clean_box,
        "multimask": multimask,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
def evaluation_report(
    result: Mapping[str, Any],
    reference_mask: Any = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With a boolean ``reference_mask`` of the same shape as the returned masks the report carries one
    ``mask_iou`` entry per candidate as sample-sanity geometry evidence; without one the verdict is
    ``not-measurable`` and the report says what labelled data would make the task measurable.
    """
    masks = np.asarray(result["masks"])
    scores = [float(v) for v in result["iou_scores"]]
    best = int(np.argmax(scores)) if scores else None
    base = {
        "task": "promptable single-object image segmentation (point and/or box prompts)",
        "decision_rule": (
            "keep the candidate with the highest model-predicted IoU; the pipeline ships no "
            "acceptance threshold and does not choose for the caller"
        ),
        "score_semantics": (
            "iou_scores are the model's own uncalibrated predicted IoU for each candidate, not a "
            "measured overlap and not a probability; the regression head is unclipped, so a value "
            "may exceed 1.0"
        ),
        "sample_kind": sample_kind,
        "n_masks": int(masks.shape[0]) if masks.ndim == 3 else 0,
        "best_candidate": best,
        "iou_scores_model_predicted": scores,
        "mask_areas_px": [int(mask.sum()) for mask in masks] if masks.ndim == 3 else [],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_mask is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth mask was supplied for the evaluated image",
            "needs": (
                "hand-labelled boolean masks for the prompted objects on your own images, scored with "
                "mask_iou per object and averaged into a mean IoU over a held-out set; no labelled "
                "mask set ships with this repository"
            ),
        }
    reference = np.asarray(reference_mask)
    return {
        **base,
        "metrics": [
            {
                "id": "mask_iou",
                "candidate": index,
                "value": mask_iou(masks[index], reference),
                "selected": index == best,
                "estimation": "one reference mask on a single scene, no dispersion estimate",
            }
            for index in range(masks.shape[0])
        ],
        "reference_area_px": int(reference.sum()),
        "verdict": "sample-sanity",
        "reason": (
            "one reference mask on one tutorial sample; geometry sanity evidence, not a segmentation "
            "benchmark"
        ),
        "needs": (
            "a labelled mask set from the deployment domain for any mean-IoU or boundary-quality claim"
        ),
    }


def validate_segmentation_dataset(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Validate paired images, prompts, and boolean masks for bounded adaptation."""
    if isinstance(records, str | bytes) or not isinstance(records, Sequence):
        raise TypeError("records must be a sequence of mappings")
    if not 2 <= len(records) <= MAX_ADAPTATION_RECORDS:
        raise ValueError(f"record count {len(records)} outside 2..{MAX_ADAPTATION_RECORDS}")

    ids: list[str] = []
    digest = hashlib.sha256()
    positive_pixels = 0
    for index, record in enumerate(records):
        if not isinstance(record, Mapping):
            raise TypeError(f"record {index} must be a mapping")
        record_id = record.get("id")
        if not isinstance(record_id, str) or not record_id.strip():
            raise ValueError(f"record {index} id must be a non-empty string")
        if record_id in ids:
            raise ValueError(f"duplicate record id: {record_id}")
        ids.append(record_id)

        image = validate_image(record.get("image"))
        mask = np.asarray(record.get("mask"))
        if mask.dtype != np.bool_:
            raise TypeError(f"record {record_id} mask must be boolean")
        if mask.shape != (image.height, image.width):
            raise ValueError(
                f"record {record_id} mask shape {mask.shape} != image {(image.height, image.width)}"
            )
        area = int(mask.sum())
        if area == 0 or area == mask.size:
            raise ValueError(f"record {record_id} mask must contain foreground and background")
        clean_points, clean_labels, clean_box = validate_prompts(
            image.width,
            image.height,
            record.get("points"),
            record.get("point_labels"),
            record.get("box"),
        )
        if clean_points is not None and clean_labels is not None:
            for (x, y), label in zip(clean_points, clean_labels, strict=True):
                row = min(int(y), image.height - 1)
                column = min(int(x), image.width - 1)
                pixel_is_foreground = bool(mask[row, column])
                if pixel_is_foreground != bool(label):
                    raise ValueError(
                        f"record {record_id} point ({x}, {y}) label {label} contradicts target mask"
                    )
        positive_pixels += area
        digest.update(record_id.encode("utf-8"))
        digest.update(np.asarray(image, dtype=np.uint8).tobytes())
        digest.update(mask.tobytes())
        digest.update(
            json.dumps(
                {"points": clean_points, "point_labels": clean_labels, "box": clean_box},
                sort_keys=True,
                separators=(",", ":"),
            ).encode("utf-8")
        )

    return {
        "records": len(records),
        "unique_ids": len(ids),
        "positive_pixels": positive_pixels,
        "dataset_sha256": digest.hexdigest(),
        "verdict": "accepted",
    }


def _segmentation_record_content_sha256(record: Mapping[str, Any]) -> str:
    """Fingerprint one validated image/mask sample without its ID or mutable prompts."""
    image = validate_image(record["image"])
    digest = hashlib.sha256()
    digest.update(np.asarray(image, dtype=np.uint8).tobytes())
    digest.update(np.asarray(record["mask"], dtype=np.bool_).tobytes())
    return digest.hexdigest()


@dataclass
class SAM2SegmentationPipeline:
    """Promptable image segmentation (points/box -> masks) over the pinned SAM 2.1 Hiera-Small checkpoint.

    Image mode only: `Sam2Model` + `Sam2Processor`. Video tracking (`Sam2VideoModel`) is not exposed."""

    _runner: Callable[..., tuple[np.ndarray, list[float]]]
    device: str
    model: Any | None = None
    processor: Any | None = None
    adaptation_config: dict[str, Any] = field(default_factory=dict)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SAM2SegmentationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Sam2Model, Sam2Processor
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Sam2Processor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        # config.json declares Sam2VideoModel; image-mode Sam2Model loads the same weights (upstream README).
        model = Sam2Model.from_pretrained(source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs)
        model = model.to(resolved_device).eval()

        def runner(image, points, labels, box, multimask) -> tuple[np.ndarray, list[float]]:
            prompt_kwargs: dict[str, Any] = {}
            if points is not None:
                prompt_kwargs["input_points"] = [[points]]
                prompt_kwargs["input_labels"] = [[labels]]
            if box is not None:
                prompt_kwargs["input_boxes"] = [[box]]
            inputs = processor(images=image, return_tensors="pt", **prompt_kwargs).to(resolved_device)
            with torch.inference_mode():
                outputs = model(**inputs, multimask_output=multimask)
            masks = processor.post_process_masks(
                outputs.pred_masks.cpu(), inputs["original_sizes"].cpu(), mask_threshold=MASK_THRESHOLD
            )[0]
            return masks[0].numpy().astype(np.bool_), [float(v) for v in outputs.iou_scores[0, 0].tolist()]

        return cls(runner, resolved_device, model=model, processor=processor)

    def freeze_for_adaptation(self) -> dict[str, int]:
        """Freeze the base model and enable the first mask-token hypernetwork only."""
        if self.model is None:
            raise RuntimeError("cannot configure adaptation without an underlying torch model")
        trainable = frozen = 0
        for name, parameter in self.model.named_parameters():
            parameter.requires_grad = name.startswith(TRAINABLE_PREFIXES)
            if parameter.requires_grad:
                trainable += parameter.numel()
            else:
                frozen += parameter.numel()
        if trainable == 0:
            raise RuntimeError("SAM2 adaptation selected no trainable parameters")
        self.adaptation_config = {
            "method": "frozen-backbone-mask-hypernetwork-gradient-adaptation",
            "trainable_prefixes": list(TRAINABLE_PREFIXES),
            "trainable_parameters": trainable,
            "frozen_parameters": frozen,
        }
        return {"trainable_parameters": trainable, "frozen_parameters": frozen}

    def finetune(
        self,
        train_records: Sequence[Mapping[str, Any]],
        val_records: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 2,
        learning_rate: float = 2e-5,
        seed: int = 42,
    ) -> list[dict[str, Any]]:
        """Run bounded mask-hypernetwork fine-tuning with BCE plus soft-Dice loss."""
        import random

        import torch
        import torch.nn.functional as F
        from torch.optim import AdamW

        if self.model is None or self.processor is None:
            raise RuntimeError("cannot fine-tune without the underlying model and processor")
        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):
            raise TypeError("learning_rate must be numeric")
        if not 0 < float(learning_rate) <= 1e-2:
            raise ValueError("learning_rate must be in (0, 1e-2]")
        train_manifest = validate_segmentation_dataset(train_records)
        val_manifest = validate_segmentation_dataset(val_records) if val_records else None
        if val_records:
            train_ids = {str(record["id"]) for record in train_records}
            val_ids = {str(record["id"]) for record in val_records}
            overlapping_ids = sorted(train_ids & val_ids)
            if overlapping_ids:
                raise ValueError(
                    f"train and validation records overlap by id: {overlapping_ids[:5]}"
                )
            train_content = {
                _segmentation_record_content_sha256(record) for record in train_records
            }
            val_content = {
                _segmentation_record_content_sha256(record) for record in val_records
            }
            if train_content & val_content:
                raise ValueError("train and validation records overlap by image/mask content")
        if not self.adaptation_config:
            self.freeze_for_adaptation()
        if any(
            parameter.requires_grad and not name.startswith(TRAINABLE_PREFIXES)
            for name, parameter in self.model.named_parameters()
        ):
            raise RuntimeError("parameters outside the declared SAM2 adapter surface are trainable")

        torch.manual_seed(seed)
        device = torch.device(self.device)
        self.model.to(device).eval()
        trainable = {
            name: parameter
            for name, parameter in self.model.named_parameters()
            if parameter.requires_grad
        }
        if not trainable:
            raise RuntimeError("model has no trainable parameters")
        before = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}
        optimizer = AdamW(list(trainable.values()), lr=float(learning_rate))

        def prepare(
            record: Mapping[str, Any],
        ) -> tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]:
            prompt_kwargs: dict[str, Any] = {}
            if record.get("points") is not None:
                prompt_kwargs["input_points"] = [[record["points"]]]
                prompt_kwargs["input_labels"] = [[record["point_labels"]]]
            if record.get("box") is not None:
                prompt_kwargs["input_boxes"] = [[record["box"]]]
            batch = self.processor(
                images=record["image"], return_tensors="pt", **prompt_kwargs
            ).to(device)
            with torch.inference_mode():
                embeddings = [
                    item.detach().cpu()
                    for item in self.model.get_image_embeddings(batch["pixel_values"])
                ]
            prompts = {
                key: batch[key].detach().cpu()
                for key in ("input_points", "input_labels", "input_boxes")
                if key in batch
            }
            target = torch.from_numpy(np.asarray(record["mask"], dtype=np.float32))
            original_size = torch.tensor([[record["image"].height, record["image"].width]])
            return embeddings, prompts, target, original_size

        cached_train = [prepare(record) for record in train_records]
        cached_val = [prepare(record) for record in val_records] if val_records else []

        def score(
            cached: Sequence[
                tuple[list[torch.Tensor], dict[str, torch.Tensor], torch.Tensor, torch.Tensor]
            ],
        ) -> float:
            values: list[float] = []
            self.model.eval()
            with torch.inference_mode():
                for embeddings, prompts, target, original_size in cached:
                    device_embeddings = [item.to(device) for item in embeddings]
                    device_prompts = {key: value.to(device) for key, value in prompts.items()}
                    device_target = target.to(device)
                    outputs = self.model(
                        image_embeddings=device_embeddings,
                        multimask_output=False,
                        **device_prompts,
                    )
                    public_mask = self.processor.post_process_masks(
                        outputs.pred_masks.cpu(),
                        original_size,
                        mask_threshold=MASK_THRESHOLD,
                    )[0][0, 0]
                    values.append(mask_iou(public_mask.numpy(), device_target.cpu().numpy().astype(bool)))
            return float(np.mean(values)) if values else 0.0

        history: list[dict[str, Any]] = []
        baseline_val_iou = score(cached_val) if cached_val else None
        for epoch in range(1, epochs + 1):
            order = list(range(len(cached_train)))
            random.Random(seed + epoch * 17).shuffle(order)
            total_loss = 0.0
            for index in order:
                embeddings, prompts, target, _original_size = cached_train[index]
                device_embeddings = [item.to(device) for item in embeddings]
                device_prompts = {key: value.to(device) for key, value in prompts.items()}
                device_target = target.to(device)
                optimizer.zero_grad(set_to_none=True)
                logits = self.model(
                    image_embeddings=device_embeddings,
                    multimask_output=False,
                    **device_prompts,
                ).pred_masks[0, 0, 0]
                target_low = F.interpolate(
                    device_target[None, None], size=logits.shape, mode="nearest"
                )[0, 0]
                probabilities = logits.sigmoid()
                bce = F.binary_cross_entropy_with_logits(logits, target_low)
                dice = 1.0 - (2.0 * (probabilities * target_low).sum() + 1.0) / (
                    probabilities.sum() + target_low.sum() + 1.0
                )
                loss = bce + dice
                loss.backward()
                optimizer.step()
                total_loss += float(loss.item())
            epoch_data: dict[str, Any] = {
                "epoch": epoch,
                "train_loss": round(total_loss / len(cached_train), 6),
                "optimizer_steps": len(cached_train),
            }
            if cached_val:
                epoch_data["val_mask_iou"] = round(score(cached_val), 6)
            history.append(epoch_data)

        delta_sq = 0.0
        for name, parameter in trainable.items():
            delta_sq += float(torch.sum((parameter.detach().cpu() - before[name]) ** 2).item())
        weight_delta_l2 = delta_sq**0.5
        if weight_delta_l2 == 0.0:
            raise RuntimeError("fine-tuning completed without changing adapter weights")
        self.model.eval()
        self.adaptation_config.update(
            {
                "epochs": epochs,
                "learning_rate": float(learning_rate),
                "batch_size": 1,
                "seed": seed,
                "loss": "binary-cross-entropy-plus-soft-dice",
                "train_manifest": train_manifest,
                "validation_manifest": val_manifest,
                "baseline_validation_mask_iou": baseline_val_iou,
                "history": history,
                "weight_delta_l2": weight_delta_l2,
            }
        )
        return history

    def evaluate_adaptation(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Evaluate adapted masks against ground truth and a prompt-box baseline."""
        validate_segmentation_dataset(records)
        model_ious: list[float] = []
        box_ious: list[float] = []
        box_model_ious: list[float] = []
        for record in records:
            result = self.segment(
                record["image"],
                points=record.get("points"),
                point_labels=record.get("point_labels"),
                box=record.get("box"),
                multimask=False,
            )
            target = np.asarray(record["mask"], dtype=np.bool_)
            model_iou = mask_iou(result["masks"][0], target)
            model_ious.append(model_iou)
            box = record.get("box")
            if box is not None:
                baseline = np.zeros_like(target)
                x0, y0, x1, y1 = (int(round(value)) for value in box)
                baseline[y0:y1, x0:x1] = True
                box_ious.append(mask_iou(baseline, target))
                box_model_ious.append(model_iou)
        box_baseline_mean_iou = float(np.mean(box_ious)) if box_ious else None
        box_prompt_model_mean_iou = float(np.mean(box_model_ious)) if box_model_ious else None
        return {
            "records": len(records),
            "mean_mask_iou": float(np.mean(model_ious)),
            "per_record_mask_iou": model_ious,
            "box_baseline_records": len(box_ious),
            "box_prompt_model_mean_iou": box_prompt_model_mean_iou,
            "box_baseline_mean_iou": box_baseline_mean_iou,
            "delta_over_box_baseline": (
                None
                if box_baseline_mean_iou is None
                else float(box_prompt_model_mean_iou - box_baseline_mean_iou)
            ),
        }

    def save_artifact(self, output_dir: str | Path, *, producer_revision: str) -> Path:
        """Write a safe mask-hypernetwork adapter plus a closed integrity manifest."""
        from safetensors.torch import save_file

        weight_delta = self.adaptation_config.get("weight_delta_l2")
        history = self.adaptation_config.get("history")
        if (
            self.model is None
            or not isinstance(weight_delta, int | float)
            or isinstance(weight_delta, bool)
            or weight_delta <= 0
            or not isinstance(history, list)
            or not history
        ):
            raise RuntimeError("artifact export requires completed fine-tuning with a positive weight delta")
        if len(producer_revision) != 40 or any(ch not in "0123456789abcdef" for ch in producer_revision):
            raise ValueError("producer_revision must be a lowercase 40-hex Git commit")
        root = Path(output_dir)
        root.mkdir(parents=True, exist_ok=True)
        if any(root.iterdir()):
            raise FileExistsError(f"artifact directory is not empty: {root}")
        state = {
            name: tensor.detach().cpu().contiguous()
            for name, tensor in self.model.state_dict().items()
            if name.startswith(TRAINABLE_PREFIXES)
        }
        if not state:
            raise RuntimeError("no adapter tensors selected for export")
        weights_path = root / ARTIFACT_WEIGHTS_NAME
        save_file(state, str(weights_path))
        manifest = {
            "artifactSpec": "1.0",
            "format": ARTIFACT_FORMAT,
            "formatVersion": ARTIFACT_FORMAT_VERSION,
            "artifactClass": "ADAPTER",
            "artifactKind": "sam2-mask-hypernetwork-adapter",
            "producer": {"pipelineId": "sam2-segmentation-pipeline", "revision": producer_revision},
            "createdAtUtc": datetime.now(UTC).isoformat(),
            "baseModel": {"id": MODEL_ID, "revision": MODEL_REVISION},
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "adaptation": dict(self.adaptation_config),
            "trainablePrefixes": list(TRAINABLE_PREFIXES),
            "retainedData": {"containsTrainingRecords": False, "containsSupportRecords": False},
            "serialization": "safetensors",
        }
        (root / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return root

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify and load a SAM2 adapter without code-capable deserialization."""
        from safetensors.torch import load_file

        if self.model is None:
            raise RuntimeError("cannot load an artifact without an underlying torch model")
        root = Path(artifact_dir)
        manifest_path = root / ARTIFACT_MANIFEST_NAME
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")
        expected_files = {ARTIFACT_MANIFEST_NAME, ARTIFACT_WEIGHTS_NAME}
        actual_entries = {path.name for path in root.iterdir()}
        if actual_entries != expected_files:
            raise ValueError(
                f"artifact directory must contain exactly {sorted(expected_files)}, "
                f"found {sorted(actual_entries)}"
            )
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"unrecognized artifact format: {manifest.get('format')}")
        if manifest.get("formatVersion") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(f"unsupported artifact formatVersion: {manifest.get('formatVersion')}")
        if manifest.get("baseModel") != {"id": MODEL_ID, "revision": MODEL_REVISION}:
            raise ValueError("artifact base model identity is incompatible")
        if manifest.get("trainablePrefixes") != list(TRAINABLE_PREFIXES):
            raise ValueError("artifact trainable prefixes do not match this pipeline")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must inventory exactly one weights file")
        entry = files[0]
        if entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError("artifact manifest names an unexpected weights path")
        weights_path = root / ARTIFACT_WEIGHTS_NAME
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights not found: {weights_path}")
        if weights_path.stat().st_size != entry.get("bytes") or _sha256(weights_path) != entry.get("sha256"):
            raise ValueError("artifact weights failed size or SHA-256 verification")
        adaptation = manifest.get("adaptation")
        if not isinstance(adaptation, dict):
            raise ValueError("artifact adaptation metadata must be a mapping")
        state = load_file(str(weights_path), device=self.device)
        expected = {
            name for name in self.model.state_dict() if name.startswith(TRAINABLE_PREFIXES)
        }
        if set(state) != expected:
            raise ValueError("artifact tensor inventory does not match the declared adapter surface")
        self.model.load_state_dict(state, strict=False)
        self.model.to(self.device).eval()
        self.adaptation_config = dict(adaptation)
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> SAM2SegmentationPipeline:
        """Construct a fresh pinned base model and attach a verified adapter."""
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

    def segment(
        self,
        image: Image.Image,
        *,
        points: Sequence[Sequence[float]] | None = None,
        point_labels: Sequence[int] | None = None,
        box: Sequence[float] | None = None,
        multimask: bool = True,
    ) -> dict[str, Any]:
        """Segment one object; returns K boolean masks (K = 3 with multimask, else 1) at input resolution."""
        rgb, clean_points, clean_labels, clean_box = _check_inputs(
            image, points, point_labels, box, multimask
        )
        masks, iou_scores = self._runner(rgb, clean_points, clean_labels, clean_box, multimask)
        masks = np.asarray(masks)
        expected = (NUM_MULTIMASK_OUTPUTS if multimask else 1, rgb.height, rgb.width)
        if masks.dtype != np.bool_ or masks.shape != expected or len(iou_scores) != expected[0]:
            raise RuntimeError(f"backend returned {masks.shape} {masks.dtype}, {len(iou_scores)} scores")
        return {
            "masks": masks,
            "iou_scores": [float(v) for v in iou_scores],
            "multimask": multimask,
            "points": clean_points,
            "point_labels": clean_labels,
            "box": clean_box,
            "width": rgb.width,
            "height": rgb.height,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `ee5bba1d82bb…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `SAM2SegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [3]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "sam2.1-hiera-small",
  "modelId": "facebook/sam2.1-hiera-small",
  "revision": "ee5bba1d82bb8749febdf90f45e84b687142ba03",
  "files": [
    {
      "path": "README.md",
      "bytes": 19974,
      "sha256": "6ec2d54879e41ad876d8cded0d641e8bc6ab74d5e095a73afc3f8406372ee6e9"
    },
    {
      "path": "config.json",
      "bytes": 5698,
      "sha256": "97ff9f65b76d107acda4247885f0a5555d0048850ae3c5f97183df289aaecde9"
    },
    {
      "path": "model.safetensors",
      "bytes": 184305280,
      "sha256": "0a4067b11ce1e23d5229203f11c718a823060d15a4b23fa2372a7d4b77cbbc60"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 683,
      "sha256": "6ebf229ee259368ce4a8d4f2fe893a72b053023710853e257253939e601f583d"
    },
    {
      "path": "processor_config.json",
      "bytes": 95,
      "sha256": "f8a68e865cfad115c1c2763f3d93eca7b1c622da06da2a9273eb437fb2389b6d"
    },
    {
      "path": "sam2.1_hiera_s.yaml",
      "bytes": 3761,
      "sha256": "632e5cd0104f5ab6cd4f9d2dfd80a8e7240e481ad7960a13cad2ae3504b88dbd"
    },
    {
      "path": "video_preprocessor_config.json",
      "bytes": 705,
      "sha256": "9fccfe5f464ec38c2f236d0e6a68e95511c80c22132fc2fa4b9f7b65f24fad95"
    }
  ],
  "totalBytes": 184336196
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = WEIGHTS_DIR / MANIFEST_NAME
if manifest_path.is_file():
    with open(manifest_path, encoding='utf-8') as handle:
        existing_manifest = json.load(handle)
    if existing_manifest != MANIFEST:
        raise RuntimeError('existing snapshot manifest differs from the inline pinned manifest')
else:
    with open(manifest_path, 'w', encoding='utf-8') as handle:
        json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = SAM2SegmentationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'facebook/sam2.1-hiera-small', 'revision': 'ee5bba1d82bb8749febdf90f45e84b687142ba03', 'license': 'apache-2.0', 'files': 7, 'total_bytes': 184336196}


README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/184M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

...

video_preprocessor_config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

{'weights_dir': '/content/weights/sam2.1-hiera-small', 'fetched': ['README.md', 'config.json', 'model.safetensors', 'preprocessor_config.json', 'processor_config.json', 'sam2.1_hiera_s.yaml', 'video_preprocessor_config.json']}


{'verified_files': 7, 'revision': 'ee5bba1d82bb8749febdf90f45e84b687142ba03'}


You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. Generate or upload a paired segmentation dataset

The default 24 deterministic 256×256 scenes contain varied rounded rectangles and ellipses plus distractors. Each record has an RGB image, boolean mask, foreground point, and bounding box. Records 0–17 train; 18–23 are held out. BYOD uses matching `images/<id>` and `masks/<id>.png` members and rejects unsafe archive paths before decoding.

In [4]:
import io
import zipfile

import numpy as np
from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}

def record_from_pair(record_id, image, mask_image):
    image = image.convert('RGB')
    mask = np.asarray(mask_image.convert('L')) > 127
    ys, xs = np.where(mask)
    if len(xs) == 0:
        raise ValueError(f'{record_id} has an empty mask')
    x0, x1, y0, y1 = int(xs.min()), int(xs.max()) + 1, int(ys.min()), int(ys.max()) + 1
    point = [float(np.median(xs)), float(np.median(ys))]
    if not mask[int(point[1]), int(point[0])]:
        point = [float(xs[0]), float(ys[0])]
    return {'id': record_id, 'image': image, 'mask': mask, 'points': [point], 'point_labels': [1], 'box': [float(x0), float(y0), float(x1), float(y1)]}

def generated_records(start=0, count=24):
    records = []
    for index in range(start, start + count):
        rng = np.random.default_rng(8100 + index)
        image = Image.new('RGB', (256, 256), tuple(int(v) for v in rng.integers(35, 95, 3)))
        draw = ImageDraw.Draw(image)
        mask_image = Image.new('L', image.size, 0)
        mask_draw = ImageDraw.Draw(mask_image)
        x0, y0 = int(rng.integers(28, 105)), int(rng.integers(28, 105))
        width, height = int(rng.integers(70, 125)), int(rng.integers(65, 120))
        box = [x0, y0, min(244, x0 + width), min(244, y0 + height)]
        colour = tuple(int(v) for v in rng.integers(145, 245, 3))
        if index % 2:
            draw.ellipse(box, fill=colour); mask_draw.ellipse(box, fill=255)
        else:
            radius = 18 + index % 12
            draw.rounded_rectangle(box, radius=radius, fill=colour)
            mask_draw.rounded_rectangle(box, radius=radius, fill=255)
        distractor = [int(rng.integers(5, 55)), int(rng.integers(175, 215)), int(rng.integers(70, 120)), int(rng.integers(225, 250))]
        draw.rectangle(distractor, fill=tuple(int(v) for v in rng.integers(90, 180, 3)))
        records.append(record_from_pair(f'generated-{index:02d}', image, mask_image))
    return records

def records_from_zip(blob):
    if len(blob) > 256 * 1024 * 1024:
        raise ValueError('BYOD ZIP exceeds the 256 MiB upload ceiling')
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        infos = archive.infolist()
        names = archive.namelist()
        normalized = [name.replace('\\', '/') for name in names]
        if len(names) > 130 or sum(info.file_size for info in infos) > 512 * 1024 * 1024:
            raise ValueError('unsafe or oversized BYOD archive')
        if any(info.flag_bits & 1 for info in infos) or any(name.startswith('/') or '..' in name.split('/') for name in normalized):
            raise ValueError('unsafe or oversized BYOD archive')
        image_entries = [(name.split('/')[-1].rsplit('.', 1)[0], original) for name, original in zip(normalized, names) if name.startswith('images/') and name.lower().endswith(('.png', '.jpg', '.jpeg'))]
        mask_entries = [(name.split('/')[-1].rsplit('.', 1)[0], original) for name, original in zip(normalized, names) if name.startswith('masks/') and name.lower().endswith('.png')]
        images, masks = dict(image_entries), dict(mask_entries)
        if len(images) != len(image_entries) or len(masks) != len(mask_entries):
            raise ValueError('BYOD archive contains duplicate image or mask stems')
        if set(images) != set(masks):
            raise ValueError('BYOD image and mask stems must match exactly')
        records = []
        for key in sorted(images):
            image = Image.open(io.BytesIO(archive.read(images[key])))
            mask = Image.open(io.BytesIO(archive.read(masks[key])))
            if min(image.size) < MIN_IMAGE_SIDE or max(image.size) > MAX_IMAGE_SIDE or min(mask.size) < MIN_IMAGE_SIDE or max(mask.size) > MAX_IMAGE_SIDE or image.size != mask.size:
                raise ValueError(f'{key} image/mask dimensions are mismatched or outside {MIN_IMAGE_SIDE}..{MAX_IMAGE_SIDE} px')
            image.load(); mask.load()
            records.append(record_from_pair(key, image, mask))
        return records

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    dataset_records = records_from_zip(next(iter(uploaded.values())))
    dataset_kind = 'BYOD'
else:
    dataset_records = generated_records()
    dataset_kind = 'generated'
if len(dataset_records) < 8:
    raise ValueError('E2E adaptation requires at least 8 records')
split_at = max(2, int(len(dataset_records) * 0.75))
train_records, val_records = dataset_records[:split_at], dataset_records[split_at:]
print({'dataset_kind': dataset_kind, 'records': len(dataset_records), 'train': len(train_records), 'held_out': len(val_records)})

{'dataset_kind': 'generated', 'records': 24, 'train': 18, 'held_out': 6}


## 5. Validate the dataset and split

Both splits must have unique IDs, exact image/mask alignment, boolean non-empty masks, valid prompts, and deterministic content fingerprints. Cross-split IDs are rejected.

In [5]:
import json
import os

os.makedirs('outputs', exist_ok=True)
train_manifest = validate_segmentation_dataset(train_records)
val_manifest = validate_segmentation_dataset(val_records)
overlap = set(r['id'] for r in train_records) & set(r['id'] for r in val_records)
if overlap:
    raise RuntimeError(f'train/validation leakage: {sorted(overlap)}')
dataset_manifest = {'kind': dataset_kind, 'train': train_manifest, 'validation': val_manifest, 'overlap_ids': []}
with open('outputs/sam2_segmentation_dataset_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(dataset_manifest, handle, indent=2)
print(json.dumps(dataset_manifest, indent=2))

{
  "kind": "generated",
  "train": {
    "records": 18,
    "unique_ids": 18,
    "positive_pixels": 132413,
    "dataset_sha256": "07134d0eb00237a703e610777b353301ea7a6c9101d1c49754d1a4b1a97c2da6",
    "verdict": "accepted"
  },
  "validation": {
    "records": 6,
    "unique_ids": 6,
    "positive_pixels": 42961,
    "dataset_sha256": "b9dd707b3f2f5029955dd64c520a5901892adcf7c346523ce47e2f08a207e7ee",
    "verdict": "accepted"
  },
  "overlap_ids": []
}


## 6. Measure the pretrained held-out baseline

Before any update, score the exact holdout and compare with the prompt-box baseline.

In [6]:
base_eval = pipe.evaluate_adaptation(val_records)
print({key: round(value, 6) if isinstance(value, float) else value for key, value in base_eval.items() if key != 'per_record_mask_iou'})

{'records': 6, 'mean_mask_iou': 0.998254, 'box_baseline_records': 6, 'box_prompt_model_mean_iou': 0.998254, 'box_baseline_mean_iou': 0.856681, 'delta_over_box_baseline': 0.141573}


## 7. Freeze the base and run bounded fine-tuning

Only the first mask-token output hypernetwork is trainable. AdamW runs two epochs with batch size one. A non-zero L2 weight delta is mandatory.

In [7]:
if not torch.cuda.is_available():
    raise RuntimeError('The canonical SAM2 E2E path requires a CUDA GPU')
parameter_counts = pipe.freeze_for_adaptation()
history = pipe.finetune(train_records, val_records, epochs=2, learning_rate=2e-5, seed=42)
print({'parameters': parameter_counts, 'history': history, 'weight_delta_l2': pipe.adaptation_config['weight_delta_l2']})

{'parameters': {'trainable_parameters': 139808, 'frozen_parameters': 38398769}, 'history': [{'epoch': 1, 'train_loss': 0.007102, 'optimizer_steps': 18, 'val_mask_iou': 0.998383}, {'epoch': 2, 'train_loss': 0.005357, 'optimizer_steps': 18, 'val_mask_iou': 0.998457}], 'weight_delta_l2': 0.10881877112418228}


## 8. Evaluate the adapted model

Score the untouched holdout after training. The report is sample-sanity, not benchmark evidence.

In [8]:
adapted_eval = pipe.evaluate_adaptation(val_records)
split_estimation = 'fixed generated held-out split' if dataset_kind == 'generated' else 'caller-provided BYOD held-out split'
evaluation_report_e2e = {'task': 'promptable-image-segmentation-adaptation', 'verdict': 'sample-sanity', 'dataset_kind': dataset_kind, 'estimation': split_estimation, 'baseline_pretrained': base_eval, 'adapted': adapted_eval, 'weight_delta_l2': pipe.adaptation_config['weight_delta_l2']}
with open('outputs/sam2_segmentation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(evaluation_report_e2e, handle, indent=2)
print(json.dumps({'baseline_mean_iou': base_eval['mean_mask_iou'], 'adapted_mean_iou': adapted_eval['mean_mask_iou'], 'box_baseline_mean_iou': adapted_eval['box_baseline_mean_iou']}, indent=2))

{
  "baseline_mean_iou": 0.9982536177479225,
  "adapted_mean_iou": 0.9984573627246697,
  "box_baseline_mean_iou": 0.8566809267647031
}


## 9. Infer on an unseen scene

A separately seeded generated record outside the train/validation index range exercises adapted serving.

In [9]:
unseen = generated_records(start=24, count=1)[0]
unseen_result = pipe.segment(unseen['image'], points=unseen['points'], point_labels=unseen['point_labels'], box=unseen['box'], multimask=False)
unseen_iou = mask_iou(unseen_result['masks'][0], unseen['mask'])
print({'id': unseen['id'], 'mask_iou_sample_sanity': unseen_iou, 'model_iou_score': unseen_result['iou_scores'][0]})

{'id': 'generated-24', 'mask_iou_sample_sanity': 1.0, 'model_iou_score': 0.9924384951591492}


## 10. Export and verify a fresh reload

Export SafeTensors plus a closed manifest. A fresh pinned base verifies the artifact; masks must match exactly and scores within the stated tolerance.

In [10]:
artifact_dir = pipe.save_artifact('outputs/sam2-mask-hypernetwork-adapter-v1', producer_revision=NOTEBOOK_SOURCE['repository_revision'])
reloaded_pipe = SAM2SegmentationPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR)
reloaded_result = reloaded_pipe.segment(unseen['image'], points=unseen['points'], point_labels=unseen['point_labels'], box=unseen['box'], multimask=False)
if not np.array_equal(unseen_result['masks'], reloaded_result['masks']):
    raise RuntimeError('reloaded adapter changed the mask')
if not np.allclose(unseen_result['iou_scores'], reloaded_result['iou_scores'], rtol=1e-5, atol=1e-6):
    raise RuntimeError('reloaded adapter changed the score beyond tolerance')
reload_summary = {'verification': 'PASSED', 'mask_exact': True, 'score_rtol': 1e-5, 'score_atol': 1e-6, 'artifact_dir': str(artifact_dir)}
print(reload_summary)

You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


{'verification': 'PASSED', 'mask_exact': True, 'score_rtol': 1e-05, 'score_atol': 1e-06, 'artifact_dir': 'outputs/sam2-mask-hypernetwork-adapter-v1'}


## 11. Export provenance and terminal summary

Bind dataset fingerprints, hyperparameters, metrics, weight activity, base identity, runtime, and reload evidence without retaining records.

In [11]:
payload = {'dataset': dataset_manifest, 'adaptation': pipe.adaptation_config, 'evaluation': evaluation_report_e2e, 'unseen': {'id': unseen['id'], 'mask_iou': unseen_iou}, 'artifact': reload_summary, 'notebook_source': NOTEBOOK_SOURCE, 'repository_revision': NOTEBOOK_SOURCE['repository_revision'], 'base_model': {'id': MODEL_ID, 'revision': MODEL_REVISION}, 'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device}}
with open('outputs/sam2_segmentation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2)
print({'status': 'E2E COMPLETE', 'train_records': len(train_records), 'held_out_records': len(val_records), 'optimizer_steps': sum(item['optimizer_steps'] for item in history), 'weight_delta_l2': pipe.adaptation_config['weight_delta_l2'], 'reload': reload_summary['verification'], 'outputs': sorted(os.listdir('outputs'))})

{'status': 'E2E COMPLETE', 'train_records': 18, 'held_out_records': 6, 'optimizer_steps': 36, 'weight_delta_l2': 0.10881877112418228, 'reload': 'PASSED', 'outputs': ['sam2-mask-hypernetwork-adapter-v1', 'sam2_segmentation_dataset_manifest.json', 'sam2_segmentation_evaluation_report.json', 'sam2_segmentation_result.json']}


## Interpretation and limits

This run proves the exact notebook can validate paired masks, update the declared adapter surface, score a held-out generated split, serialize only the adapter, attach it to the exact pinned base, and reproduce inference after reload. It does not prove improvement on photographs or any deployment domain. Real use requires rights-cleared representative images, human-reviewed masks, leakage-safe splits, and boundary-error analysis.

Successful execution proves that the recorded repository revision can complete this bounded tutorial without the repository being reachable at runtime. It does **not** establish benchmark superiority or production fitness.

## References

- Repository: https://github.com/kurtvalcorza/sam2-segmentation-pipeline
- Repository model card: https://github.com/kurtvalcorza/sam2-segmentation-pipeline/blob/main/MODEL_CARD.md
- Upstream model: https://huggingface.co/facebook/sam2.1-hiera-small
- SAM 2 paper: https://arxiv.org/abs/2408.00714